# 🛰️ SatQuery AI - Task Router Fine-Tuning (Member 2)
### Fine-Tuning a Domain-Specific Remote Sensing Query Router using QLoRA

This notebook adapts a Small Language Model (**Phi-3-mini-4k-instruct** or **Qwen2.5-3B-Instruct**) on **Google Colab's Free T4 GPU (16 GB VRAM)** to translate non-expert natural language queries into strict `TaskSpec` JSON schemas for SatQuery AI's downstream satellite pipelines (Members 1 & 3).

**Hardware Required**: Make sure your Colab runtime is set to **GPU (T4)**:
*Go to `Runtime` -> `Change runtime type` -> select `T4 GPU`*.

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers datasets peft trl bitsandbytes accelerate pydantic

## 2. Check GPU Setup

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    raise SystemError("Please change Colab runtime to GPU (Runtime -> Change runtime type -> T4 GPU)!")

## 3. Upload or Fetch Training Data
Upload the generated `train_chatml.jsonl` and `val_chatml.jsonl` files from your repository's `task_router/fine_tuning/data/` folder.

In [ ]:
import os
from google.colab import files

os.makedirs("data", exist_ok=True)

if not os.path.exists("data/train_chatml.jsonl"):
    print("Please upload your 'train_chatml.jsonl' and 'val_chatml.jsonl' files:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(filename, f"data/{filename}")

print("Files in data/:", os.listdir("data"))

## 4. Load Base Model & Apply 4-Bit QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Recommended: Phi-3-mini (3.8B) or Qwen2.5-1.5B/3B
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

print(f"[*] Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print(f"[*] Loading base model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["o_proj", "qkv_proj", "gate_up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Prepare Dataset for SFTTrainer

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "data/train_chatml.jsonl",
    "validation": "data/val_chatml.jsonl"
})

print("Train samples:", len(dataset["train"]))
print("Val samples:  ", len(dataset["validation"]))

## 6. Train with TRL SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./satquery_task_router_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    logging_dir="./logs",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args
)

print("[*] Training started...")
trainer.train()
print("[✓] Training completed successfully!")

## 7. Save Adapter & Test Inference

In [ ]:
OUTPUT_DIR = "./satquery_task_router_lora"
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model adapter saved to {OUTPUT_DIR}")

# Test query
test_query = "Find flooded areas under clouds."
prompt = f"<|system|>\nYou are SatQuery AI's Specialist Task Router. Output ONLY valid TaskSpec JSON.<|end|>\n<|user|>\n{test_query}<|end|>\n<|assistant|>\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=False)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Fine-Tuned Model Output ---")
print(response)

## 8. Download Trained LoRA Weights for Local Use
This zips the adapter files (`adapter_model.safetensors`, `adapter_config.json`) and downloads them to your local computer so your SatQuery repository can use them.

In [ ]:
import shutil
from google.colab import files

# Zip adapter
shutil.make_archive("satquery_task_router_lora", "zip", OUTPUT_DIR)
print("Downloading satquery_task_router_lora.zip...")
files.download("satquery_task_router_lora.zip")